# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [5]:
# Desarrollo de la actividad 1
TIPO_ENTIDAD = "Tipo: Municipio / Isla / Área no municipalizada"

divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()
divipola["tipo_entidad"] = divipola[TIPO_ENTIDAD].str.strip()

tipos = divipola["tipo_entidad"].value_counts()
display(tipos.to_frame("entidades"))

dimension = divipola[[
    "codigo_municipio", "Nombre Departamento", "Nombre Municipio", "tipo_entidad"
]].copy()

assert len(dimension) == len(divipola)
assert dimension["codigo_municipio"].ne("").all()
assert dimension["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all()
assert dimension["codigo_municipio"].is_unique
print("Entidades:", len(dimension))
print("Tipos:", tipos.to_dict())
print("Código de 5 cifras, sin vacíos y único: OK")
display(dimension.head())

,entidades
tipo_entidad,
Municipio,1103
Área no municipalizada,18
Isla,1


Entidades: 1122
Tipos: {'Municipio': 1103, 'Área no municipalizada': 18, 'Isla': 1}
Código de 5 cifras, sin vacíos y único: OK


,codigo_municipio,Nombre Departamento,Nombre Municipio,tipo_entidad
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio


**Interpretación del resultado:**

`dimension` conserva las **1.122 entidades territoriales** de DIVIPOLA. Cada fila tiene el código de cinco cifras (`codigo_municipio`), el nombre del departamento, el nombre del municipio y el tipo de entidad. La tabla tiene una fila por entidad, que es lo que se necesita para usarla como **dimensión** (catálogo de referencia).

| Tipo de entidad | Cantidad |
|---|---:|
| Municipio | 1.103 |
| Área no municipalizada | 18 |
| Isla | 1 |
| **Total** | **1.122** |

Las 18 **áreas no municipalizadas** son territorios, sobre todo en la Amazonía y la Orinoquía, que no están organizados como municipio pero tienen su propio código. La **isla** es una entidad insular con un código propio. Por eso el catálogo no es solo de municipios y se conserva la columna `tipo_entidad`, que permite distinguirlos.

Los `assert` confirman tres condiciones sobre `codigo_municipio`:

- **Sin vacíos.**
- **Exactamente cinco dígitos**: los dos primeros son el departamento (`05` = Antioquia) y los tres siguientes la entidad (`05001` = Medellín). Como se leyó como texto, se conserva el cero inicial.
- **Único**: ningún código aparece dos veces.

La unicidad es la condición clave: permite unir EVA con esta tabla mediante un `merge` `many_to_one` (muchas filas de EVA por cada código de DIVIPOLA) sin duplicar filas.

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [6]:
# Desarrollo de la actividad 2
total_eva = len(eva)
codigo_txt = eva["Código Dane municipio"].str.strip()
codigo_valido = codigo_txt.str.fullmatch(r"[0-9]{1,5}")

eva["codigo_municipio"] = codigo_txt.where(codigo_valido, pd.NA).str.zfill(5)
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
eva["codigo_en_revision"] = ~codigo_valido

print("Códigos vacíos:", int(codigo_txt.eq("").sum()))
print("Códigos mal formados (no vacíos):", int((codigo_txt.ne("") & ~codigo_valido).sum()))
print("Códigos completados con ceros:", int((codigo_valido & codigo_txt.str.len().lt(5)).sum()))
print("Año no interpretable:", int(eva["anio"].isna().sum()))
display(eva.loc[eva["codigo_en_revision"], ["Código Dane municipio", "Año"]].head())

eva_codigo_anio = eva[["codigo_municipio", "anio"]].drop_duplicates()
comparacion = eva_codigo_anio.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)
assert len(comparacion) == len(eva_codigo_anio)
sin_correspondencia = (
    comparacion.loc[comparacion["_merge"].eq("left_only"), ["codigo_municipio", "anio"]]
    .sort_values(["codigo_municipio", "anio"])
    .reset_index(drop=True)
)

print("Combinaciones código–año distintas:", len(eva_codigo_anio))
print("Códigos municipales distintos en EVA:", eva["codigo_municipio"].nunique())
print("Combinaciones sin correspondencia en DIVIPOLA:", len(sin_correspondencia))
display(sin_correspondencia)

Códigos vacíos: 0
Códigos mal formados (no vacíos): 0
Códigos completados con ceros: 0
Año no interpretable: 0


,Código Dane municipio,Año


Combinaciones código–año distintas: 7691
Códigos municipales distintos en EVA: 1103
Combinaciones sin correspondencia en DIVIPOLA: 0


,codigo_municipio,anio


**Interpretación del resultado:**

En EVA se construyó `codigo_municipio` a partir de `Código Dane municipio`, aplicando una regla: solo se aceptan textos numéricos de 1 a 5 cifras, y a esos se les completan ceros a la izquierda hasta llegar a cinco (por ejemplo, `5001` → `05001`). Los vacíos o mal formados quedarían como `NA` y marcados en `codigo_en_revision`, en lugar de convertirse en un código falso como `"00000"`.

En este corte el campo está limpio:

| Control | Resultado |
|---|---:|
| Códigos vacíos | 0 |
| Códigos mal formados | 0 |
| Códigos completados con ceros | 0 |
| Año no interpretable | 0 |

Que no se haya completado ningún código indica que EVA ya trae los códigos con **cinco cifras y el cero inicial** (`05001`), en el mismo formato que DIVIPOLA. Esto pasa porque se leyó con `dtype=str`; si se hubiera leído como número, los códigos de Antioquia y otros departamentos que empiezan por `0` habrían perdido ese cero y la unión habría fallado.

EVA tiene **1.103 códigos municipales distintos** y **7.691 combinaciones distintas de código y año**. Si cada código apareciera en los siete años (2019–2025), habría 1.103 × 7 = 7.721 combinaciones; la diferencia de 30 indica que algunos municipios no tienen registros en algunos años.

El **anti-join** (`merge` con `how="left"` e `indicator=True`, quedándose con las filas `left_only`) entre esas combinaciones y `dimension` da **0 combinaciones sin correspondencia**: todos los códigos municipales de EVA existen en el catálogo DIVIPOLA. La tabla `sin_correspondencia` se muestra vacía, pero conserva sus columnas como evidencia del control.

El **año se conserva solo como información** para ubicar en el tiempo un posible faltante. No se usa como clave de unión, porque DIVIPOLA es un único corte de 2024 y no tiene año. Que todos los códigos coincidan no prueba que los límites territoriales hayan sido los mismos durante 2019–2025; esa limitación se retoma en la actividad 6.

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [7]:
# Desarrollo de la actividad 3
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")

es_papa = eva["Cultivo"].str.strip().eq("Papa")
if not es_papa.any():
    print("Variantes que contienen 'papa':",
          eva.loc[eva["Cultivo"].str.contains("papa", case=False, na=False), "Cultivo"].unique())

papa_2024 = eva.loc[(es_papa & eva["anio"].eq(2024)).fillna(False).astype(bool)].copy()

cond_apta = (papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0))
apta = papa_2024.loc[cond_apta].copy()
excluidas = papa_2024.loc[~cond_apta]

motivos_exclusion = pd.Series({
    "área ausente": int(excluidas["area_ha"].isna().sum()),
    "área = 0 o negativa": int(excluidas["area_ha"].le(0).sum()),
    "producción ausente": int(excluidas["produccion_t"].isna().sum()),
    "producción negativa": int(excluidas["produccion_t"].lt(0).sum()),
}, name="filas")

print(f"papa_2024: {len(papa_2024):,} | apta: {len(apta):,} | excluidas: {len(excluidas):,}")
assert len(papa_2024) == len(apta) + len(excluidas)
display(motivos_exclusion.to_frame())
display(excluidas[["Código Dane municipio", "Año", "Cultivo", "Estado físico del cultivo",
                   "Área cosechada", "Producción"]].head())

papa_2024: 831 | apta: 779 | excluidas: 52


,filas
área ausente,0
área = 0 o negativa,52
producción ausente,0
producción negativa,0


,Código Dane municipio,Año,Cultivo,Estado físico del cultivo,Área cosechada,Producción
2602,05093,2024,Papa,En fresco,0.00,0.00
23281,15047,2024,Papa,En fresco,0.00,0.00
23834,15092,2024,Papa,En fresco,0.00,0.00
24094,15104,2024,Papa,En fresco,0.00,0.00
24367,15109,2024,Papa,En fresco,0.00,0.00


**Interpretación del resultado:**

Primero se convirtieron `Área cosechada` y `Producción` a las columnas numéricas `area_ha` y `produccion_t`. Luego la selección se hizo en dos pasos:

1. **`papa_2024`**: filas con `Cultivo == "Papa"` y año 2024 → **831 filas**.
2. **`apta`**: dentro de `papa_2024`, solo las filas con área cosechada > 0 y producción ≥ 0 → **779 filas**.

| Conjunto | Filas |
|---|---:|
| `papa_2024` | 831 |
| `apta` (entran al indicador) | **779** |
| Excluidas del indicador | 52 |

Se cumple **831 = 779 + 52**, así que ninguna fila se perdió ni se contó dos veces.

Las **52 exclusiones (6,3 %) tienen un único motivo: área cosechada igual a 0**. No hay áreas ausentes, producciones ausentes ni producciones negativas. Los ejemplos muestran que esas filas tienen área cosechada 0,00 y producción 0,00, en municipios de Antioquia (`05093`) y Boyacá (`15047`, `15092`, `15104`, `15109`). Son registros de papa reportados en 2024 **sin cosecha registrada** en ese periodo, por ejemplo siembras que aún no se cosechaban o cultivos perdidos.

Estas filas se excluyen **solo del indicador de rendimiento**, porque dividir la producción por un área de 0 da un resultado indefinido (infinito o `NaN`). No se reemplaza nada por cero ni se borran filas de `eva`: el registro original se conserva y la exclusión queda contada y explicada.

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [8]:
# Desarrollo de la actividad 4
claves = ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"]
rendimiento = apta.groupby(claves, as_index=False, dropna=False).agg(
    registros=("Cultivo", "size"),
    produccion_t=("produccion_t", "sum"),
    area_ha=("area_ha", "sum"),
)
rendimiento["rendimiento_t_ha"] = rendimiento["produccion_t"] / rendimiento["area_ha"]

assert rendimiento["area_ha"].gt(0).all()
assert rendimiento["registros"].sum() == len(apta)
print(f"Grupos: {len(rendimiento):,} | registros agrupados: {rendimiento['registros'].sum():,} = len(apta)")
print("Grupos con más de un registro:", int(rendimiento["registros"].gt(1).sum()))
print("Estados físicos:", rendimiento["Estado físico del cultivo"].value_counts().to_dict())

total_prod = rendimiento["produccion_t"].sum()
total_area = rendimiento["area_ha"].sum()
print(f"Rendimiento agregado (Σ producción / Σ área): {total_prod / total_area:.2f} t/ha")
print(f"Promedio simple de rendimientos municipales: {rendimiento['rendimiento_t_ha'].mean():.2f} t/ha")
display(rendimiento["rendimiento_t_ha"].describe().to_frame())
display(rendimiento.sort_values("produccion_t", ascending=False).head(10))

Grupos: 292 | registros agrupados: 779 = len(apta)
Grupos con más de un registro: 268
Estados físicos: {'En fresco': 292}
Rendimiento agregado (Σ producción / Σ área): 22.19 t/ha
Promedio simple de rendimientos municipales: 17.98 t/ha


,rendimiento_t_ha
count,292.000000
mean,17.978680
std,6.587110
min,3.000000
25%,14.363880
50%,18.000000
75%,21.724340
max,48.763699


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha
201,25873,Papa,En fresco,2024,4,465060.0,16060.0,28.957659
194,25793,Papa,En fresco,2024,4,329280.0,15220.0,21.634691
95,15740,Papa,En fresco,2024,4,168760.0,6610.0,25.531014
234,52560,Papa,En fresco,2024,4,157520.0,6970.0,22.599713
209,52001,Papa,En fresco,2024,4,139880.0,6710.0,20.846498
245,52838,Papa,En fresco,2024,4,129230.0,6210.0,20.809984
237,52585,Papa,En fresco,2024,4,90575.0,3665.0,24.713506
119,15861,Papa,En fresco,2024,4,80595.0,3835.0,21.015645
88,15632,Papa,En fresco,2024,4,75100.0,4250.0,17.670588
30,15001,Papa,En fresco,2024,4,64800.0,3430.0,18.892128



### Interpretación del resultado

Se agruparon los 779 registros de `apta` por municipio, cultivo, estado físico y año. Para cada grupo se calculó el número de registros, la producción total, el área cosechada y el rendimiento en toneladas por hectárea.

| Control | Resultado |
|---|---:|
| Grupos de municipios con papa en 2024 | 292 |
| Total de registros agrupados | 779 |
| Grupos con más de un registro | 268 |
| Estado físico | En fresco |
| Grupos con área cosechada mayor a cero | 292 |

Al revisar los resultados, se observa que los 779 registros fueron incluidos en la agrupación. Además, 268 de los 292 grupos tienen más de un registro, ya que algunos municipios reportan producción en diferentes periodos del año. Por esta razón, se sumaron primero la producción y el área cosechada para después calcular el rendimiento, evitando promediar directamente los valores individuales.

Todos los registros corresponden a papa en estado fresco, por lo que se mantiene una misma condición para comparar los rendimientos.

### Rendimiento de la producción

El rendimiento agregado fue de 22,19 t/ha, mientras que el promedio simple de los rendimientos municipales fue de 17,98 t/ha.

La diferencia se presenta porque el promedio simple asigna el mismo peso a todos los municipios, sin importar su área cosechada. En cambio, el rendimiento agregado considera la producción y el área total, dando mayor participación a los municipios con superficies cultivadas más grandes.

Para este caso, el valor de 22,19 t/ha representa el rendimiento general de la producción analizada.

### Distribución de los rendimientos

Los rendimientos municipales varían entre 3,0 y 48,8 t/ha, con una mediana de 18,0 t/ha. La mitad de los municipios presenta valores entre 14,4 y 21,7 t/ha.

El rendimiento máximo de 48,8 t/ha se encuentra bastante por encima de los demás valores, por lo que sería importante revisar ese registro para verificar si corresponde a una producción elevada o a un posible valor atípico.

### Municipios con mayor producción

El municipio con código `25873`, ubicado en Cundinamarca, presenta la mayor producción con 465.060 toneladas en 16.060 hectáreas cosechadas, alcanzando un rendimiento de 28,96 t/ha. Le sigue el municipio `25793`, también de Cundinamarca, con 329.280 toneladas.

Los diez municipios con mayor producción pertenecen a Cundinamarca, Boyacá y Nariño, departamentos que se destacan por su actividad agrícola relacionada con el cultivo de papa.

En esta actividad se utilizan los códigos municipales. Los nombres correspondientes se incorporarán en la actividad 5 mediante la unión con la tabla DIVIPOLA.


## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [9]:
# Desarrollo de la actividad 5
filas_antes = len(rendimiento)
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True,
)
assert len(rendimiento_municipal) == filas_antes
print("Filas antes:", filas_antes, "| después:", len(rendimiento_municipal))
display(rendimiento_municipal["_merge"].value_counts().to_frame("grupos"))

sin_nombre = rendimiento_municipal.loc[
    rendimiento_municipal["_merge"].eq("left_only"), ["codigo_municipio", "anio"]
]
print("Códigos sin nombre territorial:", len(sin_nombre))
display(sin_nombre)

print("Grupos por tipo de entidad:", rendimiento_municipal["tipo_entidad"].value_counts(dropna=False).to_dict())
top = (rendimiento_municipal
       .sort_values("produccion_t", ascending=False)
       [["Nombre Departamento", "Nombre Municipio", "Estado físico del cultivo",
         "produccion_t", "area_ha", "rendimiento_t_ha"]]
       .head(10))
display(top)

Filas antes: 292 | después: 292


,grupos
_merge,
both,292
left_only,0
right_only,0


Códigos sin nombre territorial: 0


,codigo_municipio,anio


Grupos por tipo de entidad: {'Municipio': 292}


,Nombre Departamento,Nombre Municipio,Estado físico del cultivo,produccion_t,area_ha,rendimiento_t_ha
201,CUNDINAMARCA,VILLAPINZÓN,En fresco,465060.0,16060.0,28.957659
194,CUNDINAMARCA,TAUSA,En fresco,329280.0,15220.0,21.634691
95,BOYACÁ,SIACHOQUE,En fresco,168760.0,6610.0,25.531014
234,NARIÑO,POTOSÍ,En fresco,157520.0,6970.0,22.599713
209,NARIÑO,PASTO,En fresco,139880.0,6710.0,20.846498
245,NARIÑO,TÚQUERRES,En fresco,129230.0,6210.0,20.809984
237,NARIÑO,PUPIALES,En fresco,90575.0,3665.0,24.713506
119,BOYACÁ,VENTAQUEMADA,En fresco,80595.0,3835.0,21.015645
88,BOYACÁ,SABOYÁ,En fresco,75100.0,4250.0,17.670588
30,BOYACÁ,TUNJA,En fresco,64800.0,3430.0,18.892128



### Interpretación del resultado

Se realizó la unión de `rendimiento` con `dimension` utilizando `codigo_municipio` como clave. Se aplicó un `left merge` para conservar todos los registros de rendimiento e incorporar los nombres de los municipios, departamentos y el tipo de entidad territorial. El resultado se almacenó en `rendimiento_municipal`.

| Control | Resultado |
|---|---:|
| Registros antes de la unión | 292 |
| Registros después de la unión | 292 |
| Registros con coincidencia (`both`) | 292 |
| Registros sin coincidencia (`left_only`) | 0 |
| Tipo de entidad | Municipio |

Al comparar los resultados, se observa que se conservaron los 292 registros originales y que todos encontraron correspondencia en la tabla territorial. Esto indica que la unión se realizó correctamente, sin generar registros adicionales ni dejar municipios sin identificar.

También se utilizó `validate="many_to_one"` para comprobar que cada código municipal tuviera una única correspondencia en `dimension`. De esta manera, se evitó que posibles códigos duplicados alteraran los resultados.

Los 292 registros corresponden a entidades clasificadas como municipios, sin presencia de áreas no municipalizadas u otros tipos de entidad.

### Municipios con mayor producción de papa en 2024

| Municipio | Departamento | Producción (t) | Área (ha) | Rendimiento (t/ha) |
|---|---|---:|---:|---:|
| Villapinzón | Cundinamarca | 465.060 | 16.060 | 28,96 |
| Tausa | Cundinamarca | 329.280 | 15.220 | 21,63 |
| Siachoque | Boyacá | 168.760 | 6.610 | 25,53 |
| Potosí | Nariño | 157.520 | 6.970 | 22,60 |
| Pasto | Nariño | 139.880 | 6.710 | 20,85 |
| Túquerres | Nariño | 129.230 | 6.210 | 20,81 |
| Pupiales | Nariño | 90.575 | 3.665 | 24,71 |
| Ventaquemada | Boyacá | 80.595 | 3.835 | 21,02 |
| Saboyá | Boyacá | 75.100 | 4.250 | 17,67 |
| Tunja | Boyacá | 64.800 | 3.430 | 18,89 |

En los resultados se observa que los diez municipios con mayor producción se encuentran en tres departamentos: Boyacá y Nariño, con cuatro municipios cada uno, y Cundinamarca, con dos.

Villapinzón registra la mayor producción, con 465.060 toneladas, seguido de Tausa con 329.280 toneladas. Ambos municipios presentan una diferencia considerable frente a Siachoque, que ocupa el tercer lugar con 168.760 toneladas.

Entre los diez municipios se acumulan 1.700.800 toneladas de papa y 72.960 hectáreas cosechadas. Villapinzón también presenta el rendimiento más alto de este grupo, con 28,96 t/ha, lo que muestra una relación favorable entre su producción y el área utilizada.

Finalmente, la incorporación de los nombres municipales facilita la identificación de las zonas con mayor producción. Aunque ahora se muestran los nombres, la unión y los cálculos se realizaron utilizando los códigos municipales para mantener la consistencia de los datos.


## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [10]:
# Desarrollo de la actividad 6
ruta_rend = OUT / "rendimiento_municipal.csv"
ruta_sin = OUT / "codigos_sin_correspondencia.csv"

rendimiento_municipal.assign(_merge=rendimiento_municipal["_merge"].astype(str)).to_csv(
    ruta_rend, index=False, encoding="utf-8-sig"
)
sin_correspondencia.to_csv(ruta_sin, index=False, encoding="utf-8-sig")

for ruta in (ruta_rend, ruta_sin):
    cabecera = pd.read_csv(ruta, nrows=0, encoding="utf-8-sig").columns.tolist()
    filas = len(pd.read_csv(ruta, encoding="utf-8-sig"))
    print(f"{ruta.name}: {filas} filas | cabeceras: {cabecera}")

rendimiento_municipal.csv: 292 filas | cabeceras: ['codigo_municipio', 'Cultivo', 'Estado físico del cultivo', 'anio', 'registros', 'produccion_t', 'area_ha', 'rendimiento_t_ha', 'Nombre Departamento', 'Nombre Municipio', 'tipo_entidad', '_merge']
codigos_sin_correspondencia.csv: 0 filas | cabeceras: ['codigo_municipio', 'anio']


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.